In [8]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한글 폰트를 준비합니다.
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from umap import UMAP
from pathlib import Path
from langchain_core.documents import Document

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지


DOCS_PATH = "../data/RAG/maple_jobs_documents.json"

In [9]:
model = SentenceTransformer('jhgan/ko-sroberta-multitask')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8736.85it/s]


```plain_text
A가 전달한 all_documents / all_chunks
                ↓
        page_content 추출
                ↓
        Embedding 모델 준비
                ↓
          문서 임베딩
                ↓
            embeddings
                ↓
       다음 Vector DB 단계로 전달
```

| 구분          | 내용                                                   |
| ----------- | ---------------------------------------------------- |
| 입력          | `all_documents` 또는 `all_chunks`                      |
| 입력 형태       | `list[Document]`                                     |
| 핵심 작업       | Document의 `page_content`를 임베딩 벡터로 변환                 |
| 출력          | `embeddings`                                         |
| 출력 형태       | 문서마다 하나의 숫자 벡터                                       |
| 다음 담당자에게 전달 | `all_documents` + `embeddings` + 사용한 embedding model |


### document 타입 확인

In [10]:
with open(DOCS_PATH, "r", encoding='utf-8') as f:
        data = json.load(f)

print(type(data))
print(type(data[0]))


<class 'list'>
<class 'dict'>


### 가져온 문서가 document 타입이 아닐 경우 변환하는 코드

In [14]:
def load_documents(file_path):
    with open(file_path, "r", encoding='utf-8') as f:
        data = json.load(f)

    documents = [
        Document(page_content=doc['page_content'], metadata=doc['metadata']) for doc in data
    ]
    return documents

guide_documents = load_documents("../data/RAG/maple_guides_documents_chunked.json")
jobs_documents = load_documents("../data/RAG/maple_jobs_documents.json")
items_documents = load_documents("../data/RAG/maple_items_documents.json")

all_chunks = (guide_documents + jobs_documents + items_documents)

### 문서 통합, 임베딩

In [17]:
def add_chunk_ids(documents):
    for i, doc in enumerate(documents):

        source = doc.metadata.get("source", "unknown")

        # -------------------------
        # Guide
        # -------------------------
        if source == "guide":
            article_id = doc.metadata.get("article_id")
            chunk_index = doc.metadata.get("chunk_index", 0)

            document_id = f"guide_{article_id}"
            chunk_id = f"{document_id}_{chunk_index}"

        # -------------------------
        # Item
        # -------------------------
        elif source == "probability item":
            name = doc.metadata.get("name", "unknown")
            table_index = doc.metadata.get("table_index", 0)
            row_index = doc.metadata.get("row_index", 0)

            document_id = f"item_{name}_{table_index}"
            chunk_index = row_index
            chunk_id = f"{document_id}_{row_index}"

        # -------------------------
        # Job
        # -------------------------
        elif source == "job":
            job_id = doc.metadata.get("job_id")

            document_id = f"job_{job_id}"
            chunk_index = 0
            chunk_id = f"{document_id}_0"

        # -------------------------
        # 예상하지 못한 데이터
        # -------------------------
        else:
            document_id = f"unknown_{i}"
            chunk_index = 0
            chunk_id = f"{document_id}_0"

        # 기존 metadata에 추가
        doc.metadata["document_id"] = document_id
        doc.metadata["chunk_id"] = chunk_id
        doc.metadata["chunk_index"] = chunk_index

    return documents

all_documents = add_chunk_ids(all_chunks)

for doc in all_documents[:5]:
    print(doc.metadata)

print(type(all_documents[0]))

{'source': 'guide', 'name': '게임 시작', 'section_title': '기초 가이드', 'article_id': 272, 'board_id': 429467337, 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/272', 'chunk_index': 0, 'document_id': 'guide_272', 'chunk_id': 'guide_272_0'}
{'source': 'guide', 'name': '게임 시작', 'section_title': '기초 가이드', 'article_id': 272, 'board_id': 429467337, 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/272', 'chunk_index': 1, 'document_id': 'guide_272', 'chunk_id': 'guide_272_1'}
{'source': 'guide', 'name': '게임 시작', 'section_title': '기초 가이드', 'article_id': 272, 'board_id': 429467337, 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/272', 'chunk_index': 2, 'document_id': 'guide_272', 'chunk_id': 'guide_272_2'}
{'source': 'guide', 'name': '게임 시작', 'section_title': '기초 가이드', 'article_id': 272, 'board_id': 429467337, 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/272', 'chunk_index': 3, 'document_id': 'guide_272', 'ch

In [ ]:

embeddings = model.encode(all_documents)

print("임베딩 배열 모양:", embeddings.shape)
display(embeddings)


ValueError: Unsupported input type: Document. Expected one of: str, dict, PIL.Image.Image, np.ndarray, torch.Tensor

In [19]:
# 1. Document 객체에서 순수 텍스트(page_content) 추출
all_texts = [doc.page_content for doc in all_documents]

# 2. 추출한 텍스트 리스트로 임베딩 생성
embeddings = model.encode(all_texts)

print("임베딩 배열 모양:", embeddings.shape)
display(embeddings)

임베딩 배열 모양: (3694, 768)


array([[-0.12771788, -0.03213648, -0.0516712 , ...,  0.71565   ,
         0.0791717 , -0.16086932],
       [-0.20836493, -0.32043317,  0.11371388, ...,  0.43987024,
        -0.29325446,  0.11217071],
       [-0.37273183, -0.40685868,  0.16233182, ...,  0.32537073,
        -0.0629423 , -0.06855948],
       ...,
       [ 0.09155523,  0.03220847,  0.514411  , ...,  0.30990005,
         0.20564012,  0.28118572],
       [ 0.04606242,  0.06783104,  0.41519374, ...,  0.40830278,
         0.2076993 ,  0.26604766],
       [ 0.07914903,  0.10575305,  0.5158395 , ...,  0.3012731 ,
         0.2366379 ,  0.20546584]], shape=(3694, 768), dtype=float32)